# Lab 04 · Decision Trees, Rules & Ensembles
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 4**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab04_Decision_Trees_and_Ensembles.ipynb)

**In this lab you will:**
- compute entropy and information gain by hand (Play Tennis)
- train, plot and prune a decision tree and read its rules
- compare a single tree with random forests and gradient boosting
- explain models with feature and permutation importance

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1 · Information gain by hand — the Play Tennis data

In [ ]:
tennis = pd.DataFrame({
    "Outlook":  ["Sunny","Sunny","Overcast","Rain","Rain","Rain","Overcast","Sunny","Sunny","Rain","Sunny","Overcast","Overcast","Rain"],
    "Temp":     ["Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"],
    "Humidity": ["High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"],
    "Wind":     ["Weak","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Strong"],
    "Play":     ["No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"]})
tennis

In [ ]:
def entropy(labels):
    p = labels.value_counts(normalize=True)
    return -(p * np.log2(p)).sum()

def information_gain(data, attribute, target="Play"):
    total = entropy(data[target])
    weighted = sum(len(part) / len(data) * entropy(part[target])
                   for _, part in data.groupby(attribute))
    return total - weighted

print(f"H(Play) = {entropy(tennis['Play']):.3f} bits")
gains = {a: information_gain(tennis, a) for a in ["Outlook", "Temp", "Humidity", "Wind"]}
pd.Series(gains).sort_values().plot(kind="barh", color="teal")
plt.title("Information gain of each attribute (root choice)"); plt.show()
gains

## 2 · A decision tree on a real data set

In [ ]:
from sklearn.datasets import load_wine
data = load_wine(as_frame=True)
df = data.frame
TARGET = "target"
print(df.shape); df.head()

In [ ]:
# ✏️ Optional: use your own data set instead of the built-in one
USE_MY_CSV = False   # change to True, run the cell, then choose a .csv file

if USE_MY_CSV:
    from google.colab import files          # works on Google Colab
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
    print("Loaded your file:", df.shape)
else:
    print("Using the built-in data set (wine).")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

X = df.drop(columns=[TARGET]).select_dtypes("number"); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print("test accuracy:", round(tree.score(X_test, y_test), 3))
plt.figure(figsize=(16, 7))
plot_tree(tree, feature_names=X.columns, class_names=[str(c) for c in sorted(y.unique())],
          filled=True, rounded=True, fontsize=9)
plt.show()

In [ ]:
# Every root-to-leaf path is an IF-THEN rule
print(export_text(tree, feature_names=list(X.columns)))

## 3 · Over-fitting: accuracy vs. depth

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train)); test_acc.append(m.score(X_test, y_test))
plt.plot(depths, train_acc, "o-", label="train"); plt.plot(depths, test_acc, "o-", label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Deeper is not always better"); plt.show()

## 4 · Ensembles: random forest and gradient boosting

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "Decision tree":     DecisionTreeClassifier(random_state=0),
    "Random forest":     RandomForestClassifier(n_estimators=300, random_state=0),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
scores = {name: cross_val_score(m, X, y, cv=cv) for name, m in models.items()}
pd.DataFrame(scores).plot(kind="box"); plt.ylabel("CV accuracy"); plt.title("Single tree vs. ensembles"); plt.show()
pd.DataFrame(scores).agg(["mean", "std"]).round(3)

## 5 · Which features matter?

In [ ]:
from sklearn.inspection import permutation_importance
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=0)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp.tail(10).plot(kind="barh", ax=axes[0], color="teal", title="Impurity importance (built-in)")
perm_s.tail(10).plot(kind="barh", ax=axes[1], color="orange", title="Permutation importance (test set)")
plt.tight_layout(); plt.show()

> 💡 For SHAP explanations run `!pip install shap` and then `shap.TreeExplainer(rf)`.

## ✏️ Exercises
1. Verify by hand that Gain(Humidity) = 0.151.
2. Use `criterion="entropy"` vs. `"gini"` — does the tree change?
3. Prune with `ccp_alpha` (try 0.01, 0.02 …) and plot test accuracy.
4. Which ensemble is best on your own CSV?